In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime as dt
import matplotlib.pyplot as py

In [ ]:
#https://medium.com/@hbuhrmann/creating-a-strava-style-power-curve-from-cycling-data-using-python-256147a1763f

#Read the datafile into a DataFrame
#The supplied files are clean, if you have data with nans or null values, you may have to do some extra cleaning up

df = pd.read_csv('c:/temp/bigcycledata.csv',dtype={'Watts':np.int,'Cadence':np.int,'HeartRate':np.int},parse_dates=True)
#df = pd.read_csv('c:/temp/cycledata.csv',parse_dates=True)

start = dt.now()
print('Start at ',start)

#Change the string datetime to a proper datetime format

df['datetime']=pd.to_datetime(df.Time)

# Set the elapsed duration between between each subsequent row 
# this would normally be 1 second, but sometimes it isn't

df['duration']=(df.datetime.shift(periods=-1)-df.datetime)

# Update NaT values to 0 seconds, as our dataframe is not allowed to contain non numeric values

df.loc[df['duration'].isnull(),'duration']=np.timedelta64(1,'s')

# Steps to create a powercurve like you will find in strava

# Turn the duration into an integer value in seconds
df['durationseconds']=np.divide(df['duration'], np.timedelta64(1, 's'))

# This value will be used to create a sensible log scale on the powercurve graph
logscale = 0.4

# Create a dataframe that lists all the instances of all the watts in the data
# plus the total time in each instance
spc=pd.DataFrame(df.groupby(['Watts']).durationseconds.sum())

# Sort by highest watts first
spc.reset_index(inplace=True)
spc.sort_values(by=['Watts'],ascending = False,inplace=True)

# Calculate work (watts x duration) + cumulative work  + cumulative average power
spc['work']=spc.Watts*spc.durationseconds
spc['cumwork']=spc.work.cumsum()
spc['cumduration']=spc.durationseconds.cumsum()
spc['averagecumpower']=spc.cumwork/spc.cumduration

# Calculate the log value to be used during graphing
spc['logcumdur']=spc.cumduration**logscale

spc.reset_index(inplace=True)

end = dt.now()

dur = round(df.durationseconds.sum()/3600.0,3)

print('  End at ',end,'\n  time taken for a ride of ',dur,' hours = ',end-start)

#with pd.option_context('display.max_rows', None, 'display.max_columns', None):
#   display(spc)